# Point Estimate from the Grid Posterior

Supports appendix subsection *Point estimate from the grid posterior*
(`app:point-estimate`).

Compares three ways to summarise a user's grid posterior for prediction:
**MAP**, **posterior mean**, and **full marginalization**.

This is the single source for the appendix's two point-estimate tables. It exports:

- `point_estimate_mae_smartvote_2023.tex` -- held-out predictive MAE vs. number of
  observed answers, at the default prior `tau^2 = 0.25`.
- `point_estimate_boundary_smartvote_2023.tex` -- per prior variance `tau^2`, with a
  model **retrained at that `tau^2`**: grid-edge fraction and in-sample train
  reconstruction MAE / accuracy, for MAP and posterior mean.

MAE and accuracy use the project metric functions (`metrics.compute_mae`,
`metrics.compute_accuracy`, `neutral_window=0.05`), matching the rest of the paper.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from ixplore import IXPLORE, metrics
from ixplore.optimization import posterior_means, posterior_maps

from src.data import load_dataset

RESULTS_DIR = Path("../../results/ixplore_design")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("../../tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

dataset = load_dataset("smartvote_2023")
reactions = dataset.train_reactions
n_users, n_items = reactions.shape
N_OBSERVED_LIST = [5, 10, 15, 30, 45, 60, 70]
print("data:", n_users, "users x", n_items, "items")


data: 1029 users x 75 items


## Held-out predictive MAE vs. number of observed answers

Fixed default prior `tau^2 = 0.25`. For each user we hold out a random subset of
answers, compute the posterior from the rest, and predict the held-out answers under
MAP, posterior mean, and full marginalization. MAE is averaged over all users.


In [2]:
xplore = IXPLORE(reactions, prior_variance=0.25)
xplore.iterate(n_iterations=10)
predictions_X = np.exp(xplore._log_p1)  # (G, n_items) - P(Y=1 | grid point)

rng = np.random.default_rng(42)
results = []
for n_obs in N_OBSERVED_LIST:
    maes_map, maes_mean, maes_post = [], [], []
    for user_idx in range(n_users):
        ur = reactions.iloc[user_idx]
        obs = rng.choice(n_items, size=n_obs, replace=False)
        hld = np.setdiff1d(np.arange(n_items), obs)
        y = ur.iloc[hld].values

        post = xplore.compute_posteriors(ur.iloc[obs])
        post_2d = post.reshape(1, -1)

        pm = xplore.predict(posterior_maps(post_2d, xplore.X))[0]
        maes_map.append(np.abs(y - pm[hld]).mean())
        mn = xplore.predict(posterior_means(post_2d, xplore.X))[0]
        maes_mean.append(np.abs(y - mn[hld]).mean())
        pp = (predictions_X * post.reshape(-1, 1)).sum(axis=0)
        maes_post.append(np.abs(y - pp[hld]).mean())

    results.append({"n_observed": n_obs,
                    "mae_map": np.mean(maes_map),
                    "mae_mean": np.mean(maes_mean),
                    "mae_post": np.mean(maes_post)})
    print(f"n_obs={n_obs:2d}  MAP {np.mean(maes_map):.4f}  "
          f"Mean {np.mean(maes_mean):.4f}  Full {np.mean(maes_post):.4f}")

acc_df = pd.DataFrame(results)
acc_df.round(4).to_csv(RESULTS_DIR / "point_estimates_accuracy.csv", index=False)

best = acc_df[["mae_map", "mae_mean", "mae_post"]].min(axis=1)
lines = [r"\begin{tabular}{rccc}", r"\toprule",
         r"$n_{\text{obs}}$ & MAP & Posterior mean & Full posterior \\", r"\midrule"]
for idx, r in acc_df.iterrows():
    def cell(v):
        s = f"{v:.4f}"
        return r"\textbf{" + s + "}" if abs(v - best.loc[idx]) < 1e-9 else s
    lines.append(f"{int(r.n_observed)} & {cell(r.mae_map)} & "
                 f"{cell(r.mae_mean)} & {cell(r.mae_post)} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", ""]
(TABLES_DIR / "point_estimate_mae_smartvote_2023.tex").write_text("\n".join(lines))
print("\n".join(lines))


ixplore - INFO - 2026-09-11 18:24:53 - IXPLORE initialized: 1029 users × 75 items (0 missing, 0.00%), grid 200×200, prior var=0.25.


ixplore - INFO - 2026-09-11 18:24:53 - Initial — mae: 0.1845, accuracy: 0.8355, spread: 0.4572, boundary: 0.0661


ixplore - INFO - 2026-09-11 18:24:58 - Fit complete after 10 iterations — mae: 0.1805, accuracy: 0.8359, spread: 0.2171, boundary: 0.0000


n_obs= 5  MAP 0.2313  Mean 0.2255  Full 0.2439


n_obs=10  MAP 0.2111  Mean 0.2067  Full 0.2190


n_obs=15  MAP 0.2023  Mean 0.1983  Full 0.2078


n_obs=30  MAP 0.1934  Mean 0.1905  Full 0.1961


n_obs=45  MAP 0.1915  Mean 0.1894  Full 0.1933


n_obs=60  MAP 0.1894  Mean 0.1878  Full 0.1907


n_obs=70  MAP 0.1924  Mean 0.1910  Full 0.1935
\begin{tabular}{rccc}
\toprule
$n_{\text{obs}}$ & MAP & Posterior mean & Full posterior \\
\midrule
5 & 0.2313 & \textbf{0.2255} & 0.2439 \\
10 & 0.2111 & \textbf{0.2067} & 0.2190 \\
15 & 0.2023 & \textbf{0.1983} & 0.2078 \\
30 & 0.1934 & \textbf{0.1905} & 0.1961 \\
45 & 0.1915 & \textbf{0.1894} & 0.1933 \\
60 & 0.1894 & \textbf{0.1878} & 0.1907 \\
70 & 0.1924 & \textbf{0.1910} & 0.1935 \\
\bottomrule
\end{tabular}



## Prior variance: boundary fraction and train reconstruction

For each prior variance `tau^2` a **fresh model is retrained** at that `tau^2`
(`IXPLORE(reactions, prior_variance=tau); iterate(1)`). On the retrained model we
report, for MAP and posterior mean:

- **boundary fraction** -- share of embeddings within 5% of the grid edge;
- **train reconstruction MAE / accuracy** -- all answers observed (no hold-out),
  predicting every cell from each user's full-data point estimate.


In [3]:
TAU_VALUES = [0.05, 0.1, 0.25, 0.5, 1.0, 1e6]  # prior variances tau^2
boundary_threshold = 0.05

Y_true = reactions.values
obs_mask = ~np.isnan(Y_true)

brows = []
for tau in TAU_VALUES:
    m = IXPLORE(reactions, prior_variance=tau)
    m.iterate(n_iterations=10)

    lo, hi = m.limits
    margin = (hi - lo) * boundary_threshold
    def bf(emb):
        near = ((emb[:, 0] - lo < margin) | (hi - emb[:, 0] < margin) |
                (emb[:, 1] - lo < margin) | (hi - emb[:, 1] < margin))
        return near.mean()

    post = m._posteriors()
    emb_map = posterior_maps(post, m.X)
    emb_mean = posterior_means(post, m.X)

    pred_map = m.predict(emb_map)    # (N, K)
    pred_mean = m.predict(emb_mean)

    t = Y_true[obs_mask]
    brows.append({
        "tau": tau,
        "bf_map": bf(emb_map), "bf_mean": bf(emb_mean),
        "mae_map": metrics.compute_mae(t, pred_map[obs_mask]),
        "mae_mean": metrics.compute_mae(t, pred_mean[obs_mask]),
        "acc_map": metrics.compute_accuracy(t, pred_map[obs_mask], neutral_window=0.05),
        "acc_mean": metrics.compute_accuracy(t, pred_mean[obs_mask], neutral_window=0.05),
    })
    print(f"tau^2={tau:>8}  BF MAP {brows[-1]['bf_map']:.3f} Mean {brows[-1]['bf_mean']:.3f}"
          f" | MAE MAP {brows[-1]['mae_map']:.4f} Mean {brows[-1]['mae_mean']:.4f}"
          f" | ACC MAP {brows[-1]['acc_map']:.3f} Mean {brows[-1]['acc_mean']:.3f}")

bdf = pd.DataFrame(brows)
bdf.round(4).to_csv(RESULTS_DIR / "point_estimates_boundary.csv", index=False)

def sig(s):
    return r"$10^{6}$" if s >= 1e5 else f"{s:g}"

lines = [r"\begin{tabular}{rcccccc}", r"\toprule",
         r" & \multicolumn{2}{c}{Boundary frac.} & \multicolumn{2}{c}{Recon. MAE} & "
         r"\multicolumn{2}{c}{Recon. Acc.} \\",
         r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}",
         r"$\tau^2$ & MAP & Mean & MAP & Mean & MAP & Mean \\", r"\midrule"]
for _, r in bdf.iterrows():
    def lo_pair(a, b, f):
        sa, sb = f(a), f(b)
        if a < b: sa = r"\textbf{" + sa + "}"
        elif b < a: sb = r"\textbf{" + sb + "}"
        return sa, sb
    def hi_pair(a, b, f):
        sa, sb = f(a), f(b)
        if a > b: sa = r"\textbf{" + sa + "}"
        elif b > a: sb = r"\textbf{" + sb + "}"
        return sa, sb
    bfa, bfb = lo_pair(r.bf_map, r.bf_mean, lambda x: f"{x:.3f}")
    maa, mab = lo_pair(r.mae_map, r.mae_mean, lambda x: f"{x:.4f}")
    aca, acb = hi_pair(r.acc_map, r.acc_mean, lambda x: f"{x:.3f}")
    lines.append(f"{sig(r.tau)} & {bfa} & {bfb} & {maa} & {mab} & {aca} & {acb} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", ""]
(TABLES_DIR / "point_estimate_boundary_smartvote_2023.tex").write_text("\n".join(lines))
print("\n".join(lines))


ixplore - INFO - 2026-09-11 18:25:48 - IXPLORE initialized: 1029 users × 75 items (0 missing, 0.00%), grid 200×200, prior var=0.05.


ixplore - INFO - 2026-09-11 18:25:48 - Initial — mae: 0.1845, accuracy: 0.8355, spread: 0.4572, boundary: 0.0661


ixplore - INFO - 2026-09-11 18:25:53 - Fit complete after 10 iterations — mae: 0.1804, accuracy: 0.8359, spread: 0.0620, boundary: 0.0000


ixplore - INFO - 2026-09-11 18:25:53 - IXPLORE initialized: 1029 users × 75 items (0 missing, 0.00%), grid 200×200, prior var=0.1.


ixplore - INFO - 2026-09-11 18:25:53 - Initial — mae: 0.1845, accuracy: 0.8355, spread: 0.4572, boundary: 0.0661


...

ixplore - INFO - 2026-09-11 18:26:15 - IXPLORE initialized: 1029 users × 75 items (0 missing, 0.00%), grid 200×200, prior var=1000000.0.


ixplore - INFO - 2026-09-11 18:26:15 - Initial — mae: 0.1845, accuracy: 0.8355, spread: 0.4572, boundary: 0.0661


tau^2=     1.0  BF MAP 0.088 Mean 0.000 | MAE MAP 0.1811 Mean 0.1807 | ACC MAP 0.836 Mean 0.836


ixplore - INFO - 2026-09-11 18:26:20 - Fit complete after 10 iterations — mae: 0.1808, accuracy: 0.8353, spread: 0.3743, boundary: 0.0068


tau^2=1000000.0  BF MAP 0.149 Mean 0.007 | MAE MAP 0.1807 Mean 0.1808 | ACC MAP 0.835 Mean 0.835
\begin{tabular}{rcccccc}
\toprule
 & \multicolumn{2}{c}{Boundary frac.} & \multicolumn{2}{c}{Recon. MAE} & \multicolumn{2}{c}{Recon. Acc.} \\
\cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}
$\tau^2$ & MAP & Mean & MAP & Mean & MAP & Mean \\
\midrule
0.05 & 0.000 & 0.000 & 0.1824 & \textbf{0.1804} & \textbf{0.836} & 0.836 \\
0.1 & 0.000 & 0.000 & 0.1823 & \textbf{0.1804} & 0.836 & \textbf{0.836} \\
0.25 & 0.006 & \textbf{0.000} & 0.1820 & \textbf{0.1805} & 0.836 & \textbf{0.836} \\
0.5 & 0.045 & \textbf{0.000} & 0.1815 & \textbf{0.1806} & 0.836 & \textbf{0.836} \\
1 & 0.088 & \textbf{0.000} & 0.1811 & \textbf{0.1807} & \textbf{0.836} & 0.836 \\
$10^{6}$ & 0.149 & \textbf{0.007} & \textbf{0.1807} & 0.1808 & \textbf{0.835} & 0.835 \\
\bottomrule
\end{tabular}

